In [ ]:
api_key = "YOUR_API_KEY"

In [ ]:
!pip -q install pandas requests
import os
import math
import pandas as pd
import requests
from google.colab import files
os.environ["MAPBOX_ACCESS_TOKEN"] = api_key

In [ ]:
DEST_PAIRS = [
    ("nearest_pub_lat", "nearest_pub_lon", "nearest_pub_road_computed_m"),
    ("nearest_school_lat", "nearest_school_lon", "nearest_school_road_computed_m"),
    ("nearest_overground_lat", "nearest_overground_lon", "nearest_overground_road_computed_m"),
    ("nearest_underground_lat", "nearest_underground_lon", "nearest_underground_road_computed_m"),
    ("nearest_signal_lat", "nearest_signal_lon", "nearest_signal_road_computed_m"),
]

# "walking", "driving", or "cycling"
ROUTING_PROFILE = "walking"
POINTS_TO_REQUEST = 100

def haversine_m(lat1, lon1, lat2, lon2):
    r = 6371000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * r * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def mapbox_matrix_distances_m(origin_lat, origin_lon, dests, access_token, profile):
    coords = [f"{origin_lon},{origin_lat}"] + [f"{lon},{lat}" for lat, lon in dests]
    coords_str = ";".join(coords)
    destinations = ";".join(str(i) for i in range(1, len(dests) + 1))
    url = (
        f"https://api.mapbox.com/directions-matrix/v1/mapbox/{profile}/{coords_str}"
        f"?annotations=distance&sources=0&destinations={destinations}&access_token={access_token}"
    )
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    return data["distances"][0]

df = pd.read_csv("interest_points.csv")

origin_lat_col = "latitude"
origin_lon_col = "longitude"

missing = [c for c in [origin_lat_col, origin_lon_col] + [c for p in DEST_PAIRS for c in p[:2]] if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

sampled = df.sample(n=min(POINTS_TO_REQUEST, len(df)), random_state=42).copy()

token = os.getenv("MAPBOX_ACCESS_TOKEN", "").strip()
use_mapbox = bool(token)

for idx, row in sampled.iterrows():
    origin_lat = float(row[origin_lat_col])
    origin_lon = float(row[origin_lon_col])

    dests = [(float(row[lat_col]), float(row[lon_col])) for lat_col, lon_col, _ in DEST_PAIRS]

    dists = mapbox_matrix_distances_m(origin_lat, origin_lon, dests, token, ROUTING_PROFILE)
    for (_, _, dist_col), dist in zip(DEST_PAIRS, dists):
        sampled.at[idx, dist_col] = dist

output_file = "output_with_distances.csv"
sampled.to_csv(output_file, index=False)
print("Saved:", output_file)

In [ ]:
import pandas as pd

input_file = "output_with_distances.csv"
df = pd.read_csv(input_file)

pairs = [
    ("nearest_pub", "nearest_pub_road_computed_m", "nearest_pub_road_m"),
    ("nearest_school_road", "nearest_school_road_computed_m", "nearest_school_road_m"),
    ("nearest_overground_road", "nearest_overground_road_computed_m", "nearest_overground_road_m"),
    ("nearest_underground_road", "nearest_underground_road_computed_m", "nearest_underground_road_m"),
    ("nearest_signal_road", "nearest_signal_road_computed_m", "nearest_signal_road_m"),
]

# 1. Create absolute and raw difference columns
for base, computed_col, actual_col in pairs:
    if computed_col in df.columns and actual_col in df.columns:
        # Raw difference (can be negative)
        df[f"{base}_diff_m"] = df[computed_col] - df[actual_col]
        # Absolute difference (always positive)
        df[f"{base}_abs_diff_m"] = df[f"{base}_diff_m"].abs()

# 2. Compute Statistics using Absolute Differences
stats_list = []
for base, _, _ in pairs:
    abs_col = f"{base}_abs_diff_m"
    if abs_col in df.columns:
        stats_list.append({
            "Category": base,
            "Avg Abs Diff (m)": df[abs_col].mean(),
            "Min Abs Diff (m)": df[abs_col].min(),
            "Max Abs Diff (m)": df[abs_col].max()
        })

stats_df = pd.DataFrame(stats_list)

# 3. Build the display DataFrame
show_cols = []
for base, computed_col, actual_col in pairs:
    if f"{base}_diff_m" in df.columns:
        # Including both the raw and absolute diff for inspection
        show_cols.extend([computed_col, actual_col, f"{base}_diff_m", f"{base}_abs_diff_m"])

id_cols = [c for c in ["latitude", "longitude"] if c in df.columns]
result = df[id_cols + show_cols].copy()

# Save the full comparison
result.to_csv("comparison_absolute.csv", index=False)
print(f"Saved: comparison_absolute.csv\n")

# --- DISPLAY SECTION ---

# Display Absolute Summary Statistics
print("--- SUMMARY STATISTICS (ABSOLUTE TERMS) ---")
print(stats_df.to_string(index=False))
print("\n" + "="*50 + "\n")

# Settings to show ALL rows and ALL columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000) # Prevents line wrapping

print("--- ALL DATA ROWS ---")
print(result)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load your data
input_file = "output_with_distances.csv"
df = pd.read_csv(input_file)

# Define the pairs to compare
pairs = [
    ("nearest_pub", "nearest_pub_road_computed_m", "nearest_pub_road_m"),
    ("nearest_school_road", "nearest_school_road_computed_m", "nearest_school_road_m"),
    ("nearest_overground_road", "nearest_overground_road_computed_m", "nearest_overground_road_m"),
    ("nearest_underground_road", "nearest_underground_road_computed_m", "nearest_underground_road_m"),
    ("nearest_signal_road", "nearest_signal_road_computed_m", "nearest_signal_road_m"),
]

# 2. Setup the visualization (one subplot per category)
fig, axes = plt.subplots(nrows=len(pairs), ncols=1, figsize=(10, 5 * len(pairs)))

# Handle the case where there's only one pair (axes wouldn't be a list)
if len(pairs) == 1:
    axes = [axes]

# 3. Process each pair and plot the histogram
for i, (base, computed_col, actual_col) in enumerate(pairs):
    if computed_col in df.columns and actual_col in df.columns:
        # Calculate absolute difference for this specific plot
        abs_diff = (df[computed_col] - df[actual_col]).abs()

        # Plot the data
        axes[i].hist(abs_diff.dropna(), bins=50, color='skyblue', edgecolor='black')

        # Formatting
        axes[i].set_title(f'Absolute Difference Distribution: {base}', fontsize=14, fontweight='bold')
        axes[i].set_xlabel('Difference in Meters', fontsize=12)
        axes[i].set_ylabel('Number of Locations', fontsize=12)
        axes[i].grid(axis='y', alpha=0.3)

        # Optional: Annotate with Mean
        avg = abs_diff.mean()
        axes[i].axvline(avg, color='red', linestyle='dashed', linewidth=2, label=f'Avg: {avg:.2f}m')
        axes[i].legend()

# 4. Final layout and save
plt.tight_layout()
plt.savefig('differences_histograms.png')
print("Histogram saved as 'differences_histograms.png'")

# If using a Jupyter/Colab notebook, you can also show it immediately:
# plt.show()